# SSH Exec / Scan — Guided Tutorial

Clean structure • Clear explanation • Consistent print banners.

**Goal:** scan → summarize → filter actionable sessions → (optional) DANNCE validation.


In [1]:
import os, sys
from pathlib import Path
import pandas as pd
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

# Project imports (same as your original)
sys.path.append(os.path.abspath('../..'))
from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
from useful_files.sophie_check_dannce_mir_modif import dannce_valid

BANNER = lambda t: print('\n' + '='*50 + f"\n{t}\n" + '='*50)
LINE = lambda t: print('-'*4, t)
OK = lambda t='OK': print('✓', t)
WARN = lambda t: print('! ', t)


## 1) Configuration (edit here)
Keep paths explicit. Rescan threshold keeps runs fast unless forced.

In [2]:
BASE_FOLDER = "/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused"  # ← edit if needed
FAILED_PATHS_FILE = os.path.join(BASE_FOLDER, "bad_calib.txt")
FORCE_RESCAN_SESSIONS = []    # e.g. [('2024_10_18','14_41_53')]
RESCAN_THRESHOLD_DAYS = 0.001 # small → almost always rescan
BANNER("CONFIG LOADED")
print("BASE_FOLDER:", BASE_FOLDER)
print("FAILED_PATHS_FILE:", FAILED_PATHS_FILE)
print("FORCE_RESCAN_SESSIONS:", FORCE_RESCAN_SESSIONS)
print("RESCAN_THRESHOLD_DAYS:", RESCAN_THRESHOLD_DAYS)



CONFIG LOADED
BASE_FOLDER: /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused
FAILED_PATHS_FILE: /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/bad_calib.txt
FORCE_RESCAN_SESSIONS: []
RESCAN_THRESHOLD_DAYS: 0.001


## 2) Scan folder tree → Parquet logs
Idempotent by design; controlled by threshold + force list.

In [3]:
BANNER("SCANNING FOLDERS → PARQUET")
log_folder_to_parquet_sep(
    BASE_FOLDER,
    FAILED_PATHS_FILE,
    STATUS_FIELDS_CONFIG,
    force_rescan_rec_files=FORCE_RESCAN_SESSIONS,
    rescan_threshold_days=RESCAN_THRESHOLD_DAYS,
)
OK("Folder scanning complete")



SCANNING FOLDERS → PARQUET
Log for 0single5_group5_ saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5_/folder_log.parquet
Log for 0single5_group5 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5/folder_log.parquet
Log for 0single5_group5_3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5_3/folder_log.parquet
Log for extrinsics saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03_17_00/extrinsics/folder_log.parquet
Log for 0single5_group3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_17/0single5_group3/folder_log.parquet
Log for 0single4_group4 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_17/0single4_group4/folder_log.parquet
Log for 0single4_group2 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_17/0single4_group2/folder_log.parqu

## 3) Load logs and **normalize status columns**
Make filters robust to 0/1, True/False, 'YES'/'NO', or NaN.

In [4]:
BANNER("LOAD + NORMALIZE")
all_df = read_all_parquet_files(BASE_FOLDER).to_pandas()
print(f"Total rows: {len(all_df)}")
print("Columns:", sorted(all_df.columns.tolist()))

STATUS_COLS = ['mir_generate_param','sync','mini_6cam_map','dropf_handle','com','com_vis','social','dannce','dannce_vis']

def to01(s):
    # map many representations to {0,1}
    mapping = {
        True: 1, False: 0,
        'YES': 1, 'NO': 0, 'No': 0, 'Yes': 1, 'Y':1, 'N':0,
        1: 1, 0: 0, '1':1, '0':0, None: 0,
    }
    return s.map(lambda x: mapping.get(x, 0)).astype(int)

for col in STATUS_COLS:
    if col not in all_df.columns:
        all_df[col] = 0
    else:
        try:
            all_df[col] = to01(all_df[col])
        except Exception:
            # fallback: non-mapable → treat nonzero as 1
            all_df[col] = (pd.to_numeric(all_df[col], errors='coerce').fillna(0) != 0).astype(int)

OK("Status columns normalized to 0/1")



LOAD + NORMALIZE
Total rows: 35
Columns: ['after_oxytocin', 'before_oxytocin', 'calib_files', 'com', 'com_vis', 'dannce', 'dannce_vis', 'date_folder', 'dropf_handle', 'mini_6cam_map', 'mini_rec_sync', 'mini_rec_sync_com', 'miniscope', 'mir_generate_param', 'rec_file', 'rec_path', 'scan_time', 'social', 'sync', 'test']
✓ Status columns normalized to 0/1


## 4) Status summary (quick)
Straight to the point; per-column completion percentages.

In [5]:
BANNER("STATUS SUMMARY")
total = len(all_df)
for col in STATUS_COLS:
    done = int(all_df[col].sum())
    pct = (done/total*100) if total else 0
    print(f"{col:<16}: {done:4d}/{total:<4d}  ({pct:5.1f}%)")
OK()



STATUS SUMMARY
mir_generate_param:   33/35    ( 94.3%)
sync            :   24/35    ( 68.6%)
mini_6cam_map   :    0/35    (  0.0%)
dropf_handle    :    0/35    (  0.0%)
com             :   12/35    ( 34.3%)
com_vis         :   12/35    ( 34.3%)
social          :   35/35    (100.0%)
dannce          :    0/35    (  0.0%)
dannce_vis      :    0/35    (  0.0%)
✓ OK


## 5) Select actionable subset
Default: **social** sessions with **COM complete** but **COM visualization missing**.

In [8]:
BANNER("SELECT SUBSET")
mask = (all_df['social']==1) & (all_df['com']==1) & (all_df['com_vis']==1)
subset = all_df.loc[mask, ['date_folder','rec_file','social','com','com_vis']].copy()
print("Filtered sessions:", len(subset))
print(subset.head(10))
OK()



SELECT SUBSET
Filtered sessions: 12
   date_folder         rec_file  social  com  com_vis
8   2025_07_17  0single5_group4       1    1        1
10  2025_07_25  0single5_group2       1    1        1
11  2025_07_25  0single4_group5       1    1        1
15  2025_07_01  0single1_group1       1    1        1
16  2025_07_01  0single5_group5       1    1        1
17  2025_07_01  0single2_group2       1    1        1
18  2025_07_01  0single4_group4       1    1        1
19  2025_07_01  0single3_group3       1    1        1
20  2025_07_09  0single5_group2       1    1        1
21  2025_07_09  0single3_group4       1    1        1
✓ OK


## 6) (Optional) Visual timeline of COM circles
Keeps the previous aesthetic; only light deps (matplotlib).

In [9]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def visualize_sessions_timeline(df, base_folder, max_sessions=20):
    BANNER("VISUAL TIMELINE (preview)")
    rows = min(max_sessions, len(df))
    if rows == 0:
        WARN("No sessions to visualize")
    for _, r in df.head(rows).iterrows():
        img_path = Path(base_folder) / r['date_folder'] / r['rec_file'] / 'MIR_Aligned' / 'overlay_com_circle.png'
        print("-", img_path)
        if img_path.exists():
            img = mpimg.imread(img_path)
            plt.figure(figsize=(4,4)); plt.imshow(img); plt.axis('off'); plt.show()
        else:
            WARN(f"missing: {img_path}")
    OK("Timeline preview done")

visualize_sessions_timeline(subset, BASE_FOLDER, max_sessions=10)



VISUAL TIMELINE (preview)
- /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_17/0single5_group4/MIR_Aligned/overlay_com_circle.png
!  missing: /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_17/0single5_group4/MIR_Aligned/overlay_com_circle.png
- /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_25/0single5_group2/MIR_Aligned/overlay_com_circle.png
!  missing: /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_25/0single5_group2/MIR_Aligned/overlay_com_circle.png
- /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_25/0single4_group5/MIR_Aligned/overlay_com_circle.png
!  missing: /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_25/0single4_group5/MIR_Aligned/overlay_com_circle.png
- /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_01/0single1_group1/MIR_Aligned/overlay_com_circle.png
!  missing: /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/202

## 7) (Optional) DANNCE validation (parallel)
Uses your `dannce_valid()` helper, parallelized with `ProcessPoolExecutor`.

In [10]:
BANNER("DANNCE VALIDATION")
records = subset[['date_folder','rec_file']].to_dict(orient='records')

def process_dannce_session(record):
    session_path = os.path.join(BASE_FOLDER, record['date_folder'], record['rec_file'])
    try:
        dannce_valid(session_path)
        return f"✓ {session_path}"
    except Exception as e:
        return f"✗ {session_path}: {e}"

if len(records) == 0:
    WARN("No sessions selected; skip DANNCE validation")
else:
    with ProcessPoolExecutor() as ex:
        for fut in as_completed([ex.submit(process_dannce_session, r) for r in records]):
            print(fut.result())
    OK("DANNCE validation complete")



DANNCE VALIDATION
Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_09/0single3_group4'.Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_09/0single4_group5'.Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_09/0single2_group3'.Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_01/0single2_group2'.



Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_01/0single1_group1'.Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_25/0single5_group2'.Prediction file 'save_data_AVG.mat' not found in '/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_07_01/0single3_group3'.Prediction 

## 8) Final summary
Straight summary + next steps in your usual banner style.

In [11]:
BANNER("SUMMARY")
print(f"Total sessions: {len(all_df)}")
print(f"Selected subset: {len(subset)}")
print("Next steps:\n  1. Render/update COM visuals\n  2. Validate DANNCE as needed\n  3. Flip flags back to Parquet if desired")
OK()



SUMMARY
Total sessions: 35
Selected subset: 12
Next steps:
  1. Render/update COM visuals
  2. Validate DANNCE as needed
  3. Flip flags back to Parquet if desired
✓ OK
